# Qwen3 Synthetic Prompt-Injection Token Trace

安全な架空文書だけを使い、入力tokenごとのapproach投影、5本のrandom投影、応答境界でのexact候補全文score、自由生成を分けて観察します。これはprompt injection検出器ではありません。内部投影の変化、候補score、自由生成は別々の測定です。外部ツール、秘密情報、送信・削除・認証操作へ接続しないでください。

In [ ]:
!nvidia-smi
!pip -q uninstall -y gradio gradio_client
!pip -q install "transformers==4.53.2" accelerate ipywidgets matplotlib pandas

次の3ファイルを同時にアップロードしてください。

- `colab_neurostate_3axis.py`
- `colab_approach_steering_demo.py`
- `colab_prompt_injection_trace.py`

In [ ]:
from google.colab import files
from pathlib import Path
uploaded=files.upload()
for stem in ('colab_neurostate_3axis','colab_approach_steering_demo','colab_prompt_injection_trace'):
    candidates=[name for name in uploaded if name.startswith(stem) and name.endswith('.py')]
    assert candidates,f'{stem}.py がありません'
    Path(stem+'.py').write_bytes(uploaded[candidates[-1]])
print('files ready')

In [ ]:
from colab_prompt_injection_trace import PromptInjectionTraceDemo
lab=PromptInjectionTraceDemo()
print('injection trace ready')

## 条件を比較

`benign`、`injection`、同じ攻撃文を引用として含む`quoted_control`、target文字列だけを含む`target_mention_control`、targetを回答しないよう命じる`negated_target_control`、語彙量を近づけた無害な`matched_control`を順番に実行します。赤い点は注入文または対応controlの範囲です。exact candidateのAはタスク準拠要約、Bは注入成功文字列です。

In [ ]:
import ipywidgets as widgets
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display,clear_output
from colab_prompt_injection_trace import CONDITIONS,COMPLIANT_CANDIDATE,INJECTED_CANDIDATE
condition=widgets.Dropdown(options=list(CONDITIONS),value='benign',description='condition')
thinking=widgets.Checkbox(value=False,description='enable thinking',indent=False)
tokens=widgets.IntSlider(value=64,min=16,max=128,step=16,description='tokens')
button=widgets.Button(description='Run safe trace',button_style='primary')
out=widgets.Output()
def run(_):
    with out:
        clear_output(wait=True)
        spec=CONDITIONS[condition.value]
        result=lab.run_condition(condition.value,spec['prompt'],spec['marker'],COMPLIANT_CANDIDATE,INJECTED_CANDIDATE,enable_thinking=thinking.value,max_new_tokens=tokens.value)
        frame=pd.DataFrame(result.rows)
        print('condition:',result.condition)
        print('\nPrompt:\n',result.prompt)
        print('\nGenerated:\n',result.generated)
        print('\nInjection string emitted:',INJECTED_CANDIDATE in result.generated)
        display(frame)
        score_rows=[{'candidate':s.candidate,'token_ids':s.token_ids,'raw_tokens':s.raw_tokens,'baseline_total_logprob':s.baseline_total_logprob,'baseline_mean_logprob':s.baseline_mean_logprob} for s in result.candidate_scores]
        scores=pd.DataFrame(score_rows)
        print('\nExact candidate scores (observe, EOS included):')
        display(scores)
        a,b=result.candidate_scores
        print(f"compliant-injected total logprob={a.baseline_total_logprob-b.baseline_total_logprob:+.6f}")
        plt.figure(figsize=(14,5))
        plt.plot(frame['token_index'],frame['semantic_cosine'],label='semantic approach cosine',linewidth=2)
        for name in [c for c in frame.columns if c.startswith('random #') and c.endswith(' cosine')]:
            plt.plot(frame['token_index'],frame[name],alpha=.35,linewidth=1,label=name)
        marked=frame[frame['in_injection']]
        if len(marked):
            plt.axvspan(marked['token_index'].min(),marked['token_index'].max(),color='red',alpha=.12,label='injection/control span')
        plt.xlabel('input token index');plt.ylabel('cosine with source-layer direction')
        plt.title(f"Normalized input-token projections: {result.condition}")
        plt.legend(ncol=3);plt.show()
button.on_click(run)
display(condition,thinking,tokens,button,out)

## 読み方

1条件の投影が動いただけでは注入受容とは判定しません。raw projectionは特殊tokenの大きなhidden normに影響されるため、グラフはhidden normで割ったcosineを使います。4条件を比較し、(1) 注入spanでsemantic cosineがrandom群と異なるか、(2) 応答境界で準拠候補と注入候補の全文score差が変わるか、(3) 自由生成が実際に`INJECTION_ACCEPTED`を含むか、を別々に記録します。no-thinking条件でもchat templateは入力prefixへ空の`<think></think>`を置くことがありますが、これは生成された思考ではありません。既存approach holdoutとは異なる探索課題です。